# 서울시 상권변화 위험 우선순위 · 2026 현장검토 후보

**분석 목적:** 2021~2025년 서울 상권 현황으로 논문 기반 G지수(2022~2025)를 산정하고, **다음 해 G지수로 측정한 상권변화 위험의 상대적 우선순위**를 예측합니다.

- **공식 후보는 한 가지:** 분류모델 상대우선순위 상위 10%를 `집중검토 후보`로 사용합니다.
- **2026년은 실측이 아닌 추정:** 실제 피해·임대료 상승·상인 퇴출 또는 금융지원 자격을 뜻하지 않습니다.
- **회귀모델은 보조정보:** 예측 G수준과 상승폭은 공식 후보를 추가하지 않으며, 후보 내부 검토에만 사용합니다.
- **운영기준 공개:** 연도별 순위상관의 95% 부트스트랩 구간과 상위 10% 포착 lift를 계산합니다.
- **자료미비 별도 관리:** 예측 불가 상권은 안전지역으로 취급하지 않습니다.

## 실행 방법
1. 코랩에서 이 노트북을 열고 `런타임 → 모두 실행`합니다.
2. Google Drive 마운트 후 `젠트리피케이션 2021-2025` 폴더를 찾습니다.
3. 경로가 다른 경우 설정 셀의 `FOLDER_NAME` 또는 `FOLDER_ID`만 수정합니다.
4. 결과는 Drive의 `젠트리피케이션_2026_결과`에 저장됩니다.

> 단계 경계는 코드와 문서 모두 `초기: G<-1`, `주의: -1≤G<0`, `경계: 0≤G<1`, `위험: G≥1`로 통일합니다.


In [ ]:
!pip -q install folium pyproj statsmodels
import os, re, glob, zipfile, shutil, warnings, unicodedata, subprocess
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns',60)

# ── 1) 구글 드라이브 연결 ─────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── 2) 데이터 폴더 설정 (스크린샷 기준) ───────────────
FOLDER_NAME = '젠트리피케이션 2021-2025'              # 드라이브에 보이는 폴더 이름
FOLDER_ID   = '1ThRmqrUY0obZWKW0wu1xxk5XRagkKyEs'     # 주소창 folders/ 뒤의 값
MYDRIVE     = '/content/drive/MyDrive'

def nfc(s):
    # 한글 파일명은 업로드 환경(맥/윈도우)에 따라 자모가 분리돼 저장될 수 있어 통일
    return unicodedata.normalize('NFC', str(s))

def drive_id(path):
    # 코랩에 연결된 드라이브 폴더의 실제 ID 읽기 (실패하면 None)
    try:
        r = subprocess.run(['getfattr','--only-values','-n','user.drive.id',path],
                           capture_output=True, text=True)
        return r.stdout.strip() or None
    except Exception:
        return None

def locate_folder():
    # ① 이름이 정확히 같은 폴더
    exact = os.path.join(MYDRIVE, FOLDER_NAME)
    if os.path.isdir(exact):
        return exact
    # ② 이름에 '젠트리피케이션'과 '2021'이 들어간 폴더 후보 (예: '... - 복사본')
    cands = [os.path.join(MYDRIVE, d) for d in os.listdir(MYDRIVE)
             if os.path.isdir(os.path.join(MYDRIVE, d))
             and '젠트리피케이션' in nfc(d) and '2021' in nfc(d)]
    # ③ 후보 중 폴더 ID가 일치하는 것을 우선 선택
    for c in cands:
        if drive_id(c) == FOLDER_ID:
            return c
    if len(cands) == 1:
        return cands[0]
    if cands:
        print('⚠️ 후보 폴더가 여러 개입니다:', [os.path.basename(c) for c in cands])
        return sorted(cands, key=len)[0]   # 이름이 가장 짧은(원본) 폴더 사용
    return None

DRIVE_FOLDER_PATH = locate_folder()
if DRIVE_FOLDER_PATH is None:
    raise FileNotFoundError(
        f"'내 드라이브'에서 '{FOLDER_NAME}' 폴더를 찾지 못했습니다.\n"
        "· 폴더가 '공유 문서함'에 있다면: 폴더 우클릭 → 정리 → 바로가기 추가 → 내 드라이브\n"
        "· 또는 DRIVE_FOLDER_PATH = '/content/drive/MyDrive/폴더이름' 으로 직접 지정하세요.")
print('✅ 데이터 폴더:', DRIVE_FOLDER_PATH)

# ── 3) 코랩 로컬로 복사 (드라이브 직접 읽기보다 빠르고 안정적) ──
DATA_DIR = '/content/gentrification_data'
OUT_DIR  = os.path.join(MYDRIVE, '젠트리피케이션_2026_결과')
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)

n = 0
for root, dirs, names in os.walk(DRIVE_FOLDER_PATH):
    dirs[:] = [d for d in dirs if not d.startswith('.')]
    for f in names:
        if f.lower().endswith(('.csv', '.xlsx', '.xls', '.zip')):
            s = os.path.join(root, f); d = os.path.join(DATA_DIR, nfc(f))
            if not os.path.exists(d) or os.path.getsize(d) != os.path.getsize(s):
                shutil.copy2(s, d)
            n += 1
print(f'복사 완료: {n}개 파일 →', DATA_DIR)
DATA_DIRS = [DATA_DIR]   # 이후 셀은 로컬 복사본만 읽음 (중복 탐색 방지)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
⚠️ 후보 폴더가 여러 개입니다: ['젠트리피케이션 2021-2025 - 복사본', '젠트리피케이션 2021-2025 ']
✅ 데이터 폴더: /content/drive/MyDrive/젠트리피케이션 2021-2025 
복사 완료: 16개 파일 → /content/gentrification_data


In [ ]:
import os
import re
import unicodedata
import pandas as pd
from pathlib import Path

# 1. 한글 파일명 정규화
def nfc(text):
    return unicodedata.normalize("NFC", str(text)).strip()

def compact(text):
    return re.sub(r"[\s_\-()]", "", nfc(text)).lower()

# 2. Google Drive의 원본 폴더 자동 탐색
MYDRIVE = Path("/content/drive/MyDrive")

if not MYDRIVE.is_dir():
    raise FileNotFoundError(
        "Google Drive가 연결되지 않았습니다."
    )

candidates = [
    p for p in MYDRIVE.iterdir()
    if p.is_dir()
    and nfc(p.name) == "젠트리피케이션 2021-2025"
]

if len(candidates) != 1:
    raise ValueError(
        f"원본 프로젝트 폴더 확인 필요: {candidates}"
    )

DATA_ROOT = candidates[0]

# 3. 실제 파일을 가진 폴더 전부 검색
DATA_DIRS = [
    str(DATA_ROOT),
    "/content/gentrification_data"
]

def all_paths():
    paths = []

    for base in DATA_DIRS:
        if os.path.isdir(base):
            for root, dirs, names in os.walk(base):
                dirs[:] = [
                    d for d in dirs
                    if not d.startswith(".")
                ]
                for name in names:
                    if name.lower().endswith(
                        (".csv", ".xlsx", ".xls")
                    ):
                        paths.append(
                            os.path.join(root, name)
                        )

    return sorted(set(paths))

ALL_FILES = all_paths()

print("검색 대상 폴더:", DATA_DIRS)
print("검색된 파일 총수:", len(ALL_FILES))

# 4. 파일 검색 함수
def find(*words, ext=(".csv",), in_name=()):   # ★ 수정: in_name 옵션 추가
    """
    words  : 첫 단어는 파일명, 나머지는 전체 경로에서 검색
    in_name: 반드시 '파일명'에서만 찾을 단어 (예: 연도)
    """
    matches = []

    for path in ALL_FILES:
        p = Path(path)

        if p.suffix.lower() not in ext:
            continue

        filename = compact(p.stem)
        fullpath = compact(path)

        # 첫 번째 키워드는 파일명에서 찾음
        if not words or compact(words[0]) not in filename:
            continue

        # 나머지 키워드는 전체 경로에서 검색
        if not all(compact(w) in fullpath for w in words[1:]):
            continue

        # ★ 수정: 연도는 파일명에서만 검색 (폴더명 '2021-2025' 오검색 방지)
        if not all(compact(w) in filename for w in in_name):
            continue

        matches.append(str(p))

    # 동일 파일이 복사돼 있다면 원본 드라이브 우선
    by_name = {}

    for path in sorted(
        matches,
        key=lambda x: (
            0 if str(DATA_ROOT) in x else 1,
            x
        )
    ):
        key = compact(Path(path).name)
        if key not in by_name:
            by_name[key] = path

    return list(by_name.values())

# 5. 파일별 검색
YEARS = range(2021, 2026)

FILES = {
    "store": {
        y: find("점포", in_name=(f"{y}년",)) for y in YEARS      # ★ 수정
    },
    "sales": {
        y: find("추정매출", in_name=(f"{y}년",)) for y in YEARS  # ★ 수정
    },
    "flow": find("길단위인구"),
    "change": find("상권변화지표"),
    "area": find("영역"),
    "jumin": find("주민등록", ext=(".xlsx",)),
    "rent": find("골목상권분석", "임대시세")
}

# 6. 검증 결과 출력
print("\n=== 연도별 점포 및 매출 파일 ===")

problems = []

for kind in ("store", "sales"):
    for year in YEARS:
        paths = FILES[kind][year]

        print(
            f"{kind:6} {year}: {len(paths)}개",
            [nfc(Path(p).name) for p in paths]
        )

        if len(paths) != 1:
            problems.append(f"{kind} {year}: {len(paths)}개")

print("\n=== 기타 자료 ===")

for kind in ("flow", "change", "area", "jumin", "rent"):
    print(kind, len(FILES[kind]), "개")

if problems:
    raise ValueError(
        "필수 파일 검색 실패:\n" + "\n".join(problems)
    )

print("\n✅ 2021~2025년 점포·매출 파일 검색 완료")

검색 대상 폴더: ['/content/drive/MyDrive/젠트리피케이션 2021-2025 ', '/content/gentrification_data']
검색된 파일 총수: 32

=== 연도별 점포 및 매출 파일 ===
store  2021: 1개 ['서울시_상권분석서비스(점포-상권)_2021년.csv']
store  2022: 1개 ['서울시_상권분석서비스(점포-상권)_2022년.csv']
store  2023: 1개 ['서울시_상권분석서비스(점포-상권)_2023년.csv']
store  2024: 1개 ['서울시 상권분석서비스(점포-상권)_2024년.csv']
store  2025: 1개 ['서울시 상권분석서비스(점포-상권)_2025년.csv']
sales  2021: 1개 ['서울시_상권분석서비스(추정매출-상권)_2021년.csv']
sales  2022: 1개 ['서울시_상권분석서비스(추정매출-상권)_2022년.csv']
sales  2023: 1개 ['서울시_상권분석서비스(추정매출-상권)_2023년.csv']
sales  2024: 1개 ['서울시 상권분석서비스(추정매출-상권)_2024년.csv']
sales  2025: 1개 ['서울시 상권분석서비스(추정매출-상권)_2025년.csv']

=== 기타 자료 ===
flow 1 개
change 1 개
area 1 개
jumin 1 개
rent 1 개

✅ 2021~2025년 점포·매출 파일 검색 완료


## 1. 원자료 구축 및 품질 검사
원래 사용한 점포·매출·길단위인구·상권변화지표 4종을 상권×분기 단위로 결합합니다. **매출이 누락된 상권의 매출을 0으로 만들지 않습니다.**

In [ ]:
for name in ['pd', 'read_csv', 'FILES', 'YEARS']:
    print(name, '있음' if name in globals() else '없음')

pd 있음
read_csv 있음
FILES 있음
YEARS 있음


In [ ]:
def read_csv(path, **kw):
    for enc in ['utf-8-sig', 'cp949', 'euc-kr']:
        try:
            return pd.read_csv(path, encoding=enc, low_memory=False, **kw)
        except UnicodeDecodeError:
            continue
    raise ValueError(f'인코딩을 읽을 수 없음: {path}')

In [ ]:
ENG2KOR = {'stdr_yyqu_cd':'기준_년분기_코드','trdar_se_cd':'상권_구분_코드','trdar_se_cd_nm':'상권_구분_코드_명',
           'trdar_cd':'상권_코드','trdar_cd_nm':'상권_코드_명','svc_induty_cd':'서비스_업종_코드',
           'svc_induty_cd_nm':'서비스_업종_코드_명','stor_co':'점포_수','similr_induty_stor_co':'유사_업종_점포_수',
           'opbiz_rt':'개업_율','opbiz_stor_co':'개업_점포_수','clsbiz_rt':'폐업_률','clsbiz_stor_co':'폐업_점포_수',
           'frc_stor_co':'프랜차이즈_점포_수'}   # 2025년 점포 파일은 열 이름이 영어
KEY = ['기준_년분기_코드', '상권_코드']

store = pd.concat([read_csv(FILES['store'][y][0]).rename(columns=ENG2KOR) for y in YEARS])
# 유사_업종_점포_수 = 일반 점포 + 프랜차이즈 → 전체 점포 수
store = store.groupby(KEY + ['상권_구분_코드_명', '상권_코드_명'], as_index=False)[
    ['유사_업종_점포_수', '개업_점포_수', '폐업_점포_수', '프랜차이즈_점포_수']].sum().rename(columns={'유사_업종_점포_수': '전체_점포_수'})

sales = pd.concat([read_csv(FILES['sales'][y][0]) for y in YEARS])
sales = sales.groupby(KEY, as_index=False)['당월_매출_금액'].sum(min_count=1).rename(columns={'당월_매출_금액': '매출_금액'})
flow = pd.concat([read_csv(f)[KEY + ['총_유동인구_수']] for f in FILES['flow']]).drop_duplicates(KEY)
chg = pd.concat([read_csv(f)[KEY + ['운영_영업_개월_평균']] for f in FILES['change']]).drop_duplicates(KEY)

panel = (store.merge(sales, on=KEY, how='left').merge(flow, on=KEY, how='left').merge(chg, on=KEY, how='left'))
panel['연도'] = panel['기준_년분기_코드'] // 10; panel['분기'] = panel['기준_년분기_코드'] % 10
print(panel.shape); print(panel.groupby('연도')['상권_코드'].nunique().to_dict())
panel.head()

(33000, 13)
{2021: 1650, 2022: 1650, 2023: 1650, 2024: 1650, 2025: 1650}


,기준_년분기_코드,상권_코드,상권_구분_코드_명,상권_코드_명,전체_점포_수,개업_점포_수,폐업_점포_수,프랜차이즈_점포_수,매출_금액,총_유동인구_수,운영_영업_개월_평균,연도,분기
0,20211,3001491,관광특구,이태원 관광특구,1633,29,46,87,6.339763e+10,1747889.0,90,2021,1
1,20211,3001492,관광특구,명동 남대문 북창동 다동 무교동 관광특구,9148,167,227,560,3.230974e+11,5607191.0,124,2021,1
2,20211,3001493,관광특구,동대문패션타운 관광특구,15131,389,480,255,7.404989e+10,2998757.0,88,2021,1
3,20211,3001494,관광특구,종로·청계 관광특구,4754,71,79,255,2.004631e+11,7623304.0,148,2021,1
4,20211,3001495,관광특구,잠실 관광특구,3307,83,86,334,1.929257e+11,3402418.0,77,2021,1


In [ ]:
assert panel.duplicated(['상권_코드','연도','분기']).sum()==0,'중복 상권·연도·분기 존재'
print('패널 연도별 4분기 개수:')
display(pd.crosstab(panel['연도'],panel['분기']))
print('매출 결측률:',round(panel['매출_금액'].isna().mean()*100,2),'%')


패널 연도별 4분기 개수:


분기,1,2,3,4
연도,,,,
2021,1650,1650,1650,1650
2022,1650,1650,1650,1650
2023,1650,1650,1650,1650
2024,1650,1650,1650,1650
2025,1650,1650,1650,1650


매출 결측률: 4.8 %


## 2. 학술 G지수 산정
기존 2018/2022년 연구에서 제시한 AHP 가중치로 6개 코드의 가중합을 계산합니다. **연간 원자료/코드화 방식은 논문 원자료와 상이할 수 있으며 복제 연구가 아니라 공개자료 기반 응용입니다.** 2021년은 전년 대비 변화율 계산을 위한 기준연도입니다.

In [ ]:
raw = pd.read_excel(FILES['jumin'][0], header=None, dtype=str)
hdr = raw.index[raw[0].astype(str).str.contains('행정기관코드', na=False)][0]     # 머리줄 위치
yr_row = raw.iloc[hdr - 1]
year_cols = {int(re.sub(r'\D', '', str(v))): i for i, v in yr_row.items() if re.search(r'20\d\d', str(v))}
j = raw.iloc[hdr + 1:, [0, 1] + list(year_cols.values())].copy()
j.columns = ['코드10', '이름'] + list(year_cols.keys())
for y in year_cols:
    j[y] = pd.to_numeric(j[y].astype(str).str.replace(',', ''), errors='coerce')
j = j[j['코드10'].astype(str).str[5:] != '00000']              # 구 합계 행 제외
j['동그룹'] = j['코드10'].astype(str).str[:8]

MERGE = {'11230515': '11230536', '11230533': '11230536',      # 신설동·용두동 → 용신동 (2025 분동)
         '11680740': '11680675',                              # 일원2동 → 개포3동 (2022 명칭 변경)
         '11740525': '11740520', '11740526': '11740520'}      # 상일제1·2동 → 상일동 (영역 파일은 옛 코드)
j['동그룹'] = j['동그룹'].replace(MERGE)
dong = j.groupby('동그룹')[list(year_cols)].sum(min_count=1)

area = read_csv(FILES['area'][0])
area['동그룹'] = area['행정동_코드'].astype(str).replace(MERGE)
res = area[['상권_코드', '동그룹']].merge(dong, left_on='동그룹', right_index=True, how='left')
print('행정동 연결된 상권:', res[2025].notna().sum(), '/', len(res))
res = res.melt(id_vars=['상권_코드', '동그룹'], var_name='연도', value_name='거주자수'); res['연도'] = res['연도'].astype(int)

행정동 연결된 상권: 1650 / 1650


In [ ]:
W = {'상주인구':0.134,'유동인구':0.202,'창폐업':0.141,'영업기간':0.134,'프랜차이즈':0.150,'매출액':0.239}
SIGN = {'상주인구':-1,'유동인구':1,'창폐업':1,'영업기간':-1,'프랜차이즈':1,'매출액':1}
V6 = list(W)

def recode(x, winsor=0.01):
    x = x.clip(x.quantile(winsor), x.quantile(1 - winsor))
    m, s = x.mean(), x.std()
    assert s > 0, '표준편차 0: 자료 미갱신 의심'
    r = pd.Series(-2, index=x.index); r[x >= m] = -1; r[x >= m + s] = 1; r[x >= m + 2 * s] = 2
    return r

def stage(g):
    return pd.cut(g, [-np.inf, -1, 0, 1, np.inf], right=False, labels=['초기','주의','경계','위험'])

nq = panel.groupby(['상권_코드','연도'])['분기'].count().rename('n_q')
a = panel.groupby(['상권_코드','상권_코드_명','상권_구분_코드_명','연도'], as_index=False).agg(
    점포=('전체_점포_수','mean'), 개업=('개업_점포_수','sum'), 폐업=('폐업_점포_수','sum'),
    프랜=('프랜차이즈_점포_수','mean'), 매출=('매출_금액', lambda s: s.sum(min_count=4)),
    유동=('총_유동인구_수', lambda s: s.mean() if s.notna().all() else np.nan), 영업=('운영_영업_개월_평균','mean'))
a = a.merge(nq, on=['상권_코드','연도']); a.loc[a.n_q < 4, ['매출','유동']] = np.nan
a = a.merge(res[['상권_코드','연도','거주자수']], on=['상권_코드','연도'], how='left').sort_values(['상권_코드','연도'])
g = a.groupby('상권_코드')
a['상주인구'] = g['거주자수'].pct_change(fill_method=None)
a['유동인구'] = g['유동'].pct_change(fill_method=None)
a['매출액'] = (a['매출'] / a['점포']).groupby(a['상권_코드']).pct_change(fill_method=None)
a['창폐업'] = (a['개업'] + a['폐업']) / a['점포']
a['영업기간'] = a['영업']
a['프랜차이즈'] = a['프랜'] / a['점포']
a.loc[g['연도'].diff() != 1, ['상주인구','유동인구','매출액']] = np.nan

idx = a[a['연도'].between(2022, 2025)].dropna(subset=V6).copy()
idx = idx[np.isfinite(idx[V6]).all(axis=1)]
for v in V6:
    idx[v + '_코드'] = idx.groupby('연도')[v].transform(lambda x, v=v: recode(SIGN[v] * x))
idx['G지수'] = sum(W[v] * idx[v + '_코드'] for v in V6)
idx['단계'] = stage(idx['G지수'])
idx = idx.merge(area[['상권_코드','자치구_코드_명','행정동_코드_명','엑스좌표_값','와이좌표_값']], on='상권_코드', how='left')

print('연도별 단계 비율(%)')
(pd.crosstab(idx['연도'], idx['단계'], normalize='index') * 100).round(2)

연도별 단계 비율(%)


단계,초기,주의,경계,위험
연도,,,,
2022,73.20,25.32,1.41,0.06
2023,74.05,24.98,0.97,0.00
2024,76.25,22.20,1.54,0.00
2025,74.81,24.22,0.97,0.00


In [ ]:
print('학술 G지수 재현 검증:')
W_arr=np.array([W[k] for k in V6])
assert np.isclose(sum(W.values()),1)
err=(idx['G지수'].to_numpy()-idx[[v+'_코드' for v in V6]].to_numpy()@W_arr)
print('최대 산식 오차:',float(abs(err).max()))
print('포함 상권·연도 개수:',idx.groupby('연도')['상권_코드'].nunique().to_dict())
print('제외(2025):',1650-int((idx['연도']==2025).sum()))


학술 G지수 재현 검증:
최대 산식 오차: 2.220446049250313e-16
포함 상권·연도 개수: {2022: 1556, 2023: 1553, 2024: 1554, 2025: 1548}
제외(2025): 102


## 3. 다음 해 상권변화 위험 예측: 연도별 독립 검증 후 재학습

예측할 값은 **다음 해 `G지수 ≥ -1` 여부**입니다. 이는 다음 해 상권변화 위험이 주의 단계 이상인지 선별하기 위한 내부 지표이며, 실제 피해·임대료 급등·지원 자격이 아닙니다.

- 2022년 입력 → 2024년 예측 검증에 사용되는 확대창의 시작점
- 2022~2023년 입력 → 2025년 독립 검증
- 검증 종료 후 2022~2024년 입력으로 재학습 → 2026년 상대우선순위 산출
- 공식 선정규칙: 평가 가능한 서울 상권 중 상대우선순위 상위 10%


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, balanced_accuracy_score
TYPES=['골목상권','발달상권','전통시장','관광특구']
def features(df):
    x=pd.DataFrame(index=df.index)
    for v in V6:
        x[v+'_순위']=df.groupby('연도')[v].rank(pct=True)
    x['G지수']=df['G지수'];x['점포수_log']=np.log1p(df['점포'])
    for t in TYPES:x['유형_'+t]=(df['상권_구분_코드_명']==t).astype(int)
    return x
nxt=idx[['상권_코드','연도','G지수']].rename(columns={'G지수':'G_다음'}).copy()
nxt['연도']-=1
idx=idx.merge(nxt,on=['상권_코드','연도'],how='left',validate='one_to_one')
X=features(idx)
def RF():return RandomForestClassifier(n_estimators=350,min_samples_leaf=10,random_state=42,n_jobs=-1)
valid=idx.G_다음.notna()
train=idx.연도.isin([2022,2023]) & valid
test=idx.연도.eq(2024) & valid
rf=RF().fit(X.loc[train],(idx.loc[train,'G_다음']>=-1).astype(int))
yt=(idx.loc[test,'G_다음']>=-1).astype(int).to_numpy()
scores=rf.predict_proba(X.loc[test])[:,1]
base=idx.loc[test,'G지수'].to_numpy()
k=max(1,int(np.ceil(len(yt)*.1)))
def metrics(y,s):
    top=np.argsort(-s)[:k]
    precision_top=y[top].mean()
    return {'AUC':roc_auc_score(y,s),'PR-AUC':average_precision_score(y,s),
            '전체 실제주의이상률':y.mean(),'상위10% 적중률':precision_top,
            '상위10% 적중 lift':precision_top/max(y.mean(),1e-12),
            '상위10% 내 실제주의이상':int(y[top].sum())}
report=pd.DataFrame([metrics(yt,base),metrics(yt,scores)],index=['현재 G지수','RF 모델점수'])
print('2024→2025 홀드아웃 검증 표본:',len(yt),'실제 다음해 주의이상:',yt.mean())
display(report.round(3))


2024→2025 홀드아웃 검증 표본: 1547 실제 다음해 주의이상: 0.25210084033613445


,AUC,PR-AUC,전체 실제주의이상률,상위10% 적중률,상위10% 적중 lift,상위10% 내 실제주의이상
현재 G지수,0.715,0.474,0.252,0.568,2.252,88
RF 모델점수,0.770,0.605,0.252,0.774,3.071,120


In [ ]:
# 2022~2024 입력자료를 2023~2025 실측 G지수와 연결해 최종모델 학습
alltrain=idx.연도.isin([2022,2023,2024]) & valid
final=RF().fit(X.loc[alltrain],(idx.loc[alltrain,'G_다음']>=-1).astype(int))
c25=idx.연도.eq(2025)
out=idx.loc[c25,['상권_코드','상권_코드_명','상권_구분_코드_명','자치구_코드_명','행정동_코드_명',
                  '엑스좌표_값','와이좌표_값','점포','개업','폐업','프랜','G지수','단계']+V6].copy()
out['2026_상대우선순위점수']=final.predict_proba(X.loc[c25])[:,1]
out['2025_현재주의이상']=(out['G지수']>=-1)
out['2025_폐업률_참고']=np.where(out['점포']>0,out['폐업']/out['점포'],np.nan)
out=out.sort_values(['2026_상대우선순위점수','상권_코드'],ascending=[False,True]).reset_index(drop=True)
out['서울상권_우선순위']=np.arange(1,len(out)+1)
N_FINANCE=max(1,int(np.ceil(len(out)*0.10)))
out['지원검토_2단계']=np.where(out.index<N_FINANCE,'집중검토 후보','일반·관찰')
out['2026_공식집중검토']=out['지원검토_2단계'].eq('집중검토 후보')
out['근거_요약']=out.apply(lambda r:f"G2025={r['G지수']:.3f}, 2025단계={r['단계']}, 점포={r['점포']:.0f}, 폐업={r['폐업']:.0f}",axis=1)
print('2026 평가 가능 상권:',len(out),' 집중검토 후보:',N_FINANCE)
print('현재 2025 G지수 주의 이상:',int(out['2025_현재주의이상'].sum()))
display(out[['서울상권_우선순위','지원검토_2단계','자치구_코드_명','상권_코드_명','2026_상대우선순위점수','G지수','단계','점포','폐업']].head(20))


2026 평가 가능 상권: 1548  집중검토 후보: 155
현재 2025 G지수 주의 이상: 390


,서울상권_우선순위,지원검토_2단계,자치구_코드_명,상권_코드_명,2026_상대우선순위점수,G지수,단계,점포,폐업
0,1,집중검토 후보,용산구,숙대입구,0.952939,-0.373,주의,536.50,67
1,2,집중검토 후보,마포구,DMC(디지털미디어시티),0.950587,-0.441,주의,874.25,126
2,3,집중검토 후보,강동구,강일동주민센터,0.942185,-0.723,주의,209.00,31
3,4,집중검토 후보,강서구,양천향교역 7번,0.932368,-0.114,주의,388.50,70
4,5,집중검토 후보,동작구,중앙대학교,0.930688,-0.282,주의,213.75,25
5,6,집중검토 후보,동대문구,경희대학교(경희대),0.922106,-0.441,주의,411.25,62
6,7,집중검토 후보,노원구,서울과학기술대,0.921276,0.306,경계,91.00,21
7,8,집중검토 후보,금천구,독산역 1번 출입구,0.916635,-0.282,주의,69.00,8
8,9,집중검토 후보,강동구,고덕 골목형상점가,0.915437,-0.575,주의,183.00,28
9,10,집중검토 후보,노원구,경춘선숲길 우측,0.909558,-0.484,주의,351.75,44


## 추가 검증: 확대창 검증 · 상권 단위 신뢰구간 · 상권 유형 편향
**2024년 검증:** 2022년으로 학습한 뒤 2023년 입력 → 2024년 결과 확인. **2025년 검증:** 2022~2023년으로 학습한 뒤 2024년 입력 → 2025년 결과 확인. 2026년 실측 검증은 불가능합니다.

반복된 상권의 관측치를 독립 표본으로 취급하지 않도록 상권 코드 단위 부트스트랩을 수행합니다.

In [ ]:
# 추가 검증 1: 시계열 확대창 검증 (연도 간 데이터 누수 방지)
from sklearn.metrics import precision_score, recall_score, brier_score_loss
rolling_rows=[]
rolling_preds=[]
for target_year in [2024,2025]:
    test_year=target_year-1
    training_years=list(range(2022,test_year))
    tr=idx['연도'].isin(training_years) & valid
    te=idx['연도'].eq(test_year) & valid
    if not tr.any() or not te.any():continue
    model=RF().fit(X.loc[tr],(idx.loc[tr,'G_다음']>=-1).astype(int))
    prob=model.predict_proba(X.loc[te])[:,1]
    truth=(idx.loc[te,'G_다음']>=-1).astype(int).to_numpy()
    b=idx.loc[te,'G지수'].to_numpy()
    k_year=max(1,int(np.ceil(len(prob)*0.10)))
    selected=np.argsort(-prob,kind='stable')[:k_year]
    baseselect=np.argsort(-b,kind='stable')[:k_year]
    rolling_rows.append({'예측대상연도':target_year,'학습입력연도':str(training_years),
      '검증표본':len(prob),'실제주의이상비율':truth.mean(),
      'RF_AUC':roc_auc_score(truth,prob),'G지수_AUC':roc_auc_score(truth,b),
      'RF_PR_AUC':average_precision_score(truth,prob),'G지수_PR_AUC':average_precision_score(truth,b),
      'RF_상위10%적중률':truth[selected].mean(),'G지수_상위10%적중률':truth[baseselect].mean(),
      'RF_상위10%적중_lift':truth[selected].mean()/max(truth.mean(),1e-12),
      'RF_상위10%포착률':truth[selected].sum()/max(1,truth.sum()),
      '브라이어점수_참고':brier_score_loss(truth,prob)})
    fold=idx.loc[te,['상권_코드','상권_구분_코드_명','연도']].copy()
    fold['예측대상연도']=target_year;fold['실제주의이상']=truth;fold['모델점수']=prob
    fold['상위10%선정']=False
    fold.iloc[selected,fold.columns.get_loc('상위10%선정')]=True
    rolling_preds.append(fold)
rolling=pd.DataFrame(rolling_rows)
rolling_detail=pd.concat(rolling_preds,ignore_index=True)
print('=== 2024/2025년 확대창 예측 검증 ===')
display(rolling.round(4))
rolling.to_csv(os.path.join(OUT_DIR,'검증_연도별_성능.csv'),index=False,encoding='utf-8-sig')
rolling_detail.to_csv(os.path.join(OUT_DIR,'검증_연도별_상권예측.csv'),index=False,encoding='utf-8-sig')

# 추가 검증 2: 상권 코드 단위 클러스터 부트스트랩 (95% 신뢰구간)
# 동일한 상권이 여러 연도에 반복 등장하므로 상권 단위로 재표집합니다.
from sklearn.metrics import roc_auc_score, average_precision_score
rng=np.random.default_rng(2026)
unique_area=rolling_detail['상권_코드'].unique()
blocks={code:grp.index.to_numpy() for code,grp in rolling_detail.groupby('상권_코드',sort=False)}
boot=[]
for _ in range(400):
    draw=rng.choice(unique_area,size=len(unique_area),replace=True)
    bdf=rolling_detail.iloc[np.concatenate([blocks[c] for c in draw])]
    yt_b=bdf['실제주의이상'].to_numpy();sc_b=bdf['모델점수'].to_numpy()
    if np.unique(yt_b).size<2:continue
    kb=max(1,int(np.ceil(len(bdf)*.10)))
    top=np.argsort(-sc_b,kind='stable')[:kb]
    boot.append({'AUC':roc_auc_score(yt_b,sc_b),'PR_AUC':average_precision_score(yt_b,sc_b),
                 '상위10%적중률':yt_b[top].mean()})
ci=pd.DataFrame(boot)
ci_out=pd.DataFrame([{'지표':c,'CI_2.5%':ci[c].quantile(.025),'CI_97.5%':ci[c].quantile(.975)} for c in ci])
print('=== 2024/2025년 검증 통합: 상권별 클러스터 부트스트랩 95% CI ===')
display(ci_out.round(4))
ci_out.to_csv(os.path.join(OUT_DIR,'검증_클러스터부트스트랩_95CI.csv'),index=False,encoding='utf-8-sig')

# 추가 검증 3: 골목상권·발달상권·전통시장·관광특구별 선택 편향과 성능
bytype=[]
for name,grp in rolling_detail.groupby('상권_구분_코드_명',observed=True):
    yg=grp['실제주의이상'].to_numpy()
    bytype.append({'상권유형':name,'표본':len(grp),
        '실제주의이상비율':yg.mean(), '선정비율':grp['상위10%선정'].mean(),
        '선정시적중률':grp.loc[grp['상위10%선정'],'실제주의이상'].mean() if grp['상위10%선정'].any() else np.nan,
        'AUC':roc_auc_score(yg,grp['모델점수']) if len(np.unique(yg))==2 else np.nan})
bytype=pd.DataFrame(bytype).sort_values('표본',ascending=False)
print('=== 상권 유형별 편차 (적은 표본은 신중히 해석) ===')
display(bytype.round(4))
bytype.to_csv(os.path.join(OUT_DIR,'검증_유형별_편차.csv'),index=False,encoding='utf-8-sig')

=== 2024/2025년 확대창 예측 검증 ===


,예측대상연도,학습입력연도,검증표본,실제주의이상비율,RF_AUC,G지수_AUC,RF_PR_AUC,G지수_PR_AUC,RF_상위10%적중률,G지수_상위10%적중률,RF_상위10%적중_lift,RF_상위10%포착률,브라이어점수_참고
0,2024,[2022],1546,0.2354,0.7851,0.7327,0.5844,0.4648,0.7097,0.5613,3.0142,0.3022,0.1409
1,2025,"[2022, 2023]",1547,0.2521,0.7704,0.7151,0.6046,0.4739,0.7742,0.5677,3.0710,0.3077,0.1479


=== 2024/2025년 검증 통합: 상권별 클러스터 부트스트랩 95% CI ===


,지표,CI_2.5%,CI_97.5%
0,AUC,0.7542,0.7972
1,PR_AUC,0.5540,0.6353
2,상위10%적중률,0.6742,0.8065


=== 상권 유형별 편차 (적은 표본은 신중히 해석) ===


,상권유형,표본,실제주의이상비율,선정비율,선정시적중률,AUC
0,골목상권,2032,0.2554,0.0989,0.7214,0.7494
3,전통시장,551,0.2214,0.0780,0.7674,0.7799
2,발달상권,498,0.2249,0.1305,0.7846,0.8610
1,관광특구,12,0.0833,0.0833,1.0000,1.0000


## 자료미비 상권 별도 관리
매출 누락이나 인구자료 결측 등으로 G지수를 만들 수 없었던 상권은 위험이 낮다고 간주하지 않습니다. 금융지원팀에 별도 확인 목록을 제공합니다.

In [ ]:
# 자료미비 상권 별도 관리 — 분석 대상에서 제외된 상권을 '안전'으로 간주하지 않음
master=panel.loc[panel['연도']==2025,['상권_코드','상권_코드_명','상권_구분_코드_명']].drop_duplicates('상권_코드')
missing=master.loc[~master['상권_코드'].isin(out['상권_코드'])].copy()
missing=missing.merge(a.loc[a['연도']==2025,['상권_코드','점포','매출','유동','영업','거주자수']],
                      on='상권_코드',how='left',validate='one_to_one')
missing['자료상태']='2026 예측 점수 산정 불가'
missing['제외사유_참고']=missing.apply(lambda r: '매출결측/불완전' if pd.isna(r['매출']) else
  ('유동인구결측' if pd.isna(r['유동']) else ('상주인구결측' if pd.isna(r['거주자수']) else '기타 지수 입력 또는 이전 연도 결측')),axis=1)
assert len(missing)+len(out)==len(master), '2025년 전체 상권 수 검증 실패'
missing.to_csv(os.path.join(OUT_DIR,'2026_자료미비_별도검토상권.csv'),index=False,encoding='utf-8-sig')
print(f'2025년 관측된 전체 {len(master):,}개 = 2026년 평가 가능 {len(out):,}개 + 자료미비 {len(missing):,}개')
display(missing['제외사유_참고'].value_counts(dropna=False))

2025년 관측된 전체 1,650개 = 2026년 평가 가능 1,548개 + 자료미비 102개


,count
제외사유_참고,
매출결측/불완전,90
기타 지수 입력 또는 이전 연도 결측,10
유동인구결측,2


## 부록 A. 임대시세와의 탐색적 외부 비교

이 절은 **상권변화 위험 우선순위 모델의 운영 통과 여부를 결정하지 않습니다.** 행정동 임대료는 G지수와 다른 개념이며 공간 단위도 다릅니다. 결과는 향후 지표 확장을 위한 참고로만 보존합니다.

2022~2024년 G지수와 다음 해 임대료 변화의 관련성을 행정동×연도 단위로 탐색하되, 상관관계를 인과나 실제 피해 판정으로 해석하지 않습니다.


In [ ]:
# 행정동 임대시세: 외부 타당성에 대한 별도 탐색 분석
# 매칭 불가를 억지로 채우지 않으며 상관관계를 인과나 실측 피해로 해석하지 않습니다.
from scipy.stats import spearmanr
import unicodedata

def nc(s):
    return re.sub(r'[\s_()\-]', '', unicodedata.normalize('NFC', str(s))).lower()
def get_col(frame, candidates):
    lookup={nc(c):c for c in frame.columns}
    for candidate in candidates:
        if nc(candidate) in lookup: return lookup[nc(candidate)]
    return None
def code8(s):
    return s.astype('string').str.strip().str.replace(r'\.0$', '', regex=True)

if not FILES['rent']:
    print('⚠️ 임대시세 자료 부재: 외부 검증만 건너뜁니다.')
else:
    rent=read_csv(FILES['rent'][0])
    rent.columns=[unicodedata.normalize('NFC',str(c)).strip().lstrip('\ufeff') for c in rent.columns]
    rentcode=get_col(rent,['코드','행정동_코드','행정동코드'])
    areacode=get_col(area,['행정동_코드','행정동코드','코드'])
    rcol=get_col(rent,['임대료_1층','1층임대료'])
    if not rentcode or not areacode or not rcol:
        raise ValueError(f'행정동 코드 또는 임대료 컬럼을 확인하세요. 임대: {rent.columns.tolist()} / 영역: {area.columns.tolist()}')
    if '구분' in rent.columns:rent=rent[rent['구분'].astype(str).str.strip().eq('행정동')].copy()
    for col in ['연도','분기']:rent[col]=pd.to_numeric(rent[col],errors='coerce')
    rent['코드']=code8(rent[rentcode]); rent=rent[rent['코드'].str.fullmatch(r'\d{8}',na=False)].copy()
    rent['임대료_1층']=pd.to_numeric(rent[rcol].astype(str).str.replace(',','',regex=False),errors='coerce')
    rent.loc[rent['임대료_1층']<=0,'임대료_1층']=np.nan
    rent=rent[rent['연도'].between(2019,2025) & rent['분기'].between(1,4)].copy()
    # 동일 코드×연도×분기 중복이 발생하면 조용히 덮어쓰지 않고 중단
    assert not rent.duplicated(['코드','연도','분기']).any(), '임대자료 중복 코드-연도-분기'
    rent=rent.sort_values(['코드','분기','연도'])
    rent['전년값']=rent.groupby(['코드','분기'])['임대료_1층'].shift(1)
    rent['전년']=rent.groupby(['코드','분기'])['연도'].shift(1)
    rent['전년동기_상승률']=np.where(
        (rent['연도']-rent['전년']==1) & (rent['전년값']>0),
        (rent['임대료_1층']/rent['전년값']-1)*100,np.nan)
    rent_annual=rent.groupby(['코드','연도'],as_index=False)['전년동기_상승률'].median()
    rent_annual['연도']=rent_annual['연도']-1 # G(t) vs 실제 임대료 변화(t+1)
    amap=area[['상권_코드',areacode]].copy();amap['코드']=code8(amap[areacode])
    amap=amap[amap['코드'].str.fullmatch(r'\d{8}',na=False)][['상권_코드','코드']].drop_duplicates()
    ambiguous=amap.groupby('상권_코드')['코드'].nunique();excluded=ambiguous[ambiguous>1].index
    amap=amap[~amap['상권_코드'].isin(excluded)].drop_duplicates('상권_코드')
    overlap=len(set(amap['코드']) & set(rent['코드']))
    print('일치하는 행정동 코드:', overlap, '/ 다중 행정동 모호 상권 제외:',len(excluded))
    if overlap==0:raise ValueError('행정동 코드 체계 불일치: 임의 이름 매칭은 하지 않습니다.')
    compare=idx[idx['연도'].between(2022,2024)][['상권_코드','연도','G지수','점포']].merge(
        amap,on='상권_코드',how='left',validate='many_to_one').merge(
        rent_annual,on=['코드','연도'],how='left',validate='many_to_one')
    print('상권-연도 매칭률:',round(100*compare['전년동기_상승률'].notna().mean(),2),'%')
    dong_validation=compare.dropna(subset=['코드','G지수','전년동기_상승률']).groupby(
        ['코드','연도'],as_index=False).agg(G지수_평균=('G지수','mean'),
         임대료_변화=('전년동기_상승률','first'),상권수=('상권_코드','nunique'))
    if len(dong_validation)>=25 and dong_validation['G지수_평균'].nunique()>1 and dong_validation['임대료_변화'].nunique()>1:
        rho,p=spearmanr(dong_validation['G지수_평균'],dong_validation['임대료_변화'])
        print(f'임대료 외부 검증: 행정동-연도 {len(dong_validation):,}개; Spearman rho={rho:.3f}, p={p:.4f}')
        print('※ 이 p값은 동일 행정동의 연도 간 반복 관측을 완전히 보정하지 않은 탐색적 결과입니다.')
    else:print('⚠️ 표본/분산 부족으로 상관 분석 불가')
    display(dong_validation.head())
    dong_validation.to_csv(os.path.join(OUT_DIR,'외부검증_행정동연도.csv'),index=False,encoding='utf-8-sig')

일치하는 행정동 코드: 399 / 다중 행정동 모호 상권 제외: 0
상권-연도 매칭률: 99.57 %
임대료 외부 검증: 행정동-연도 1,173개; Spearman rho=0.009, p=0.7602
※ 이 p값은 동일 행정동의 연도 간 반복 관측을 완전히 보정하지 않은 탐색적 결과입니다.


,코드,연도,G지수_평균,임대료_변화,상권수
0,11110515,2022,-1.2100,-4.262168,4
1,11110515,2023,-1.6440,0.577741,4
2,11110515,2024,-1.3285,-12.557508,4
3,11110530,2022,-0.8825,-13.675876,6
4,11110530,2023,-1.4010,14.860618,6


## 5. 금융지원팀 전달 파일 + 인터랙티브 서울 지도
빨강 = 2026 집중검토 후보(예측점수 상위 10%), 회색 = 일반·관찰. 지도 오른쪽 위 레이어에서 전체 상권을 켜고 끌 수 있습니다. 각 원을 클릭하면 상권명, 모델점수, 2025년 G지수와 점포 정보를 확인할 수 있습니다. 지도는 Google Drive 결과 폴더에 HTML로 저장됩니다.


In [ ]:
from pyproj import Transformer
import folium
from folium.plugins import Fullscreen, MiniMap
from IPython.display import display
from html import escape

# 공식 선정규칙: 2026년 예측점수 상위 10%만 집중검토 (셀 12와 일치)
out['우선순위'] = out['지원검토_2단계'].map({'집중검토 후보':'집중검토 후보', '일반·관찰':'일반·관찰'})
assert int((out['우선순위']=='집중검토 후보').sum()) == N_FINANCE

# 좌표 변환 (EPSG:5181 → 위경도)
tf = Transformer.from_crs('EPSG:5181', 'EPSG:4326', always_xy=True)
out['경도'], out['위도'] = tf.transform(
    pd.to_numeric(out['엑스좌표_값'], errors='coerce').to_numpy(),
    pd.to_numeric(out['와이좌표_값'], errors='coerce').to_numpy()
)

# 결과 CSV 저장
handoff_cols = [
    '서울상권_우선순위','우선순위','지원검토_2단계','자치구_코드_명','행정동_코드_명',
    '상권_코드','상권_코드_명','상권_구분_코드_명','2026_상대우선순위점수',
    '2025_현재주의이상','G지수','단계','점포','개업','폐업',
    '2025_폐업률_참고','프랜','근거_요약','위도','경도'
]
out[handoff_cols].to_csv(os.path.join(OUT_DIR, '2026_전체상권_모델점수.csv'), index=False, encoding='utf-8-sig')
focus = out[out['우선순위'] == '집중검토 후보'].copy()
focus[handoff_cols].to_csv(os.path.join(OUT_DIR, '2026_금융지원_집중검토후보.csv'), index=False, encoding='utf-8-sig')
idx[idx['연도'] == 2025].to_csv(os.path.join(OUT_DIR, '2025_G지수_실측기준.csv'), index=False, encoding='utf-8-sig')
report.to_csv(os.path.join(OUT_DIR, '2025_검증성능.csv'), encoding='utf-8-sig')

# 서울 범위 좌표 검사
valid_geo = out['위도'].between(37.40, 37.72) & out['경도'].between(126.70, 127.25)
print(f'서울 범위 내 좌표: {int(valid_geo.sum()):,} / {len(out):,}개')
if valid_geo.sum() == 0:
    raise ValueError('지도 좌표가 서울 영역에 없습니다. 입력 좌표의 EPSG 코드를 확인하세요.')
if (~valid_geo).any():
    display(out.loc[~valid_geo, ['상권_코드','상권_코드_명','엑스좌표_값','와이좌표_값']].head(10))

# 배경지도 (API 키 없이 코랩에서 열리는 Esri 회색 지도)
ESRI_URL  = 'https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}'
ESRI_ATTR = 'Tiles &copy; Esri &mdash; Esri, HERE, Garmin, &copy; OpenStreetMap contributors'
m = folium.Map(location=[37.5665, 126.9780], zoom_start=11, tiles=None, control_scale=True)
folium.TileLayer(tiles=ESRI_URL, attr=ESRI_ATTR, name='회색 배경지도', max_zoom=16).add_to(m)

# 위험 후보 및 일반·관찰 2개 레이어
all_group  = folium.FeatureGroup(name='일반·관찰(회색)', show=False)
risk_group = folium.FeatureGroup(name='집중검토 후보 상위 10%(빨강)', show=True)
STYLE = {'집중검토 후보': ('#d4383f', 7, 0.9, risk_group),
         '일반·관찰': ('#8894a3', 2.5, 0.4, all_group)}

def popup_html(r):
    return (f'<div style="font-family:sans-serif;min-width:230px;line-height:1.7">'
            f'<b style="font-size:15px">{escape(str(r["상권_코드_명"]))}</b><br>'
            f'지역: {escape(str(r["자치구_코드_명"]))} / {escape(str(r["행정동_코드_명"]))}<br>'
            f'우선순위: <b>{escape(str(r["우선순위"]))}</b><br>'
            f'2026 예측점수: <b>{r["2026_상대우선순위점수"]:.3f}</b><br>'
            f'2025 실측 G지수: {r["G지수"]:.3f} ({escape(str(r["단계"]))})<br>'
            f'2025 평균 점포 수: {r["점포"]:.0f}<br>'
            f'2025 연간 폐업 수: {r["폐업"]:.0f}<br>'
            f'<small>주의: 예측점수는 보정된 실제 발생확률이 아닙니다.</small></div>')

for _, r in out.loc[valid_geo].iterrows():
    color, radius, opacity, layer = STYLE[r['우선순위']]
    folium.CircleMarker(
        location=[float(r['위도']), float(r['경도'])],
        radius=radius, color=color, weight=1, fill=True, fill_color=color,
        fill_opacity=opacity,
        tooltip=f"{r['상권_코드_명']} · {r['우선순위']} · 점수 {r['2026_상대우선순위점수']:.3f}",
        popup=folium.Popup(popup_html(r), max_width=340)
    ).add_to(layer)

all_group.add_to(m)    # 회색이 맨 아래
risk_group.add_to(m)   # 빨강이 맨 위
folium.LayerControl(collapsed=False).add_to(m)
Fullscreen(position='topleft').add_to(m)
MiniMap(tile_layer=folium.TileLayer(tiles=ESRI_URL, attr=ESRI_ATTR), toggle_display=True).add_to(m)

# 범례
n1 = int((out['우선순위'] == '집중검토 후보').sum())
legend = f'''<div style="position:fixed;bottom:24px;left:24px;z-index:9999;
background:white;border:1px solid #bbb;border-radius:10px;padding:12px 15px;
box-shadow:0 2px 8px #0002;font:13px/1.6 sans-serif;max-width:300px">
<b>2026 서울 상업 젠트리피케이션 위험 후보</b><br>
<span style="color:#d4383f">●</span> 집중검토 후보: 점수 상위 10% ({n1:,}개)<br>
<span style="color:#8894a3">●</span> 일반·관찰: 레이어에서 표시<br>
<small>비교 결과는 다음 해 G지수 주의 이상에 대한 예측이며, 실제 임대료 상승·퇴출·금융지원 자격을 확정하지 않습니다. 모델점수는 보정된 확률이 아닙니다.</small>
</div>'''
m.get_root().html.add_child(folium.Element(legend))

map_path = os.path.join(OUT_DIR, '2026_집중검토상권_지도.html')
m.save(map_path)
print('지도 저장:', map_path)
print(out['우선순위'].value_counts())
assert len(focus)==N_FINANCE, '금융지원 후보 수 불일치'
assert (focus['지원검토_2단계']=='집중검토 후보').all()
display(m)

서울 범위 내 좌표: 1,548 / 1,548개
지도 저장: /content/drive/MyDrive/젠트리피케이션_2026_결과/2026_집중검토상권_지도.html
우선순위
일반·관찰      1393
집중검토 후보     155
Name: count, dtype: int64


In [ ]:
# 참고용 민감도 분석. 이 기준을 2026년 공식 선정 규칙으로 사용하지 않습니다.
# 기준 점수별로 몇 개가 뽑히고, 실제로 얼마나 맞았는지 (2024→2025 검증 결과)
rows = []
for t in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    pick = scores >= t
    rows.append({'기준점수': t,
                 '검증 선정 수': int(pick.sum()),
                 '적중률(정밀도)': round(yt[pick].mean(), 3) if pick.any() else None,
                 '2026 선정 수': int((out['2026_상대우선순위점수'] >= t).sum())})
display(pd.DataFrame(rows))
print('2026 모델점수 분포:')
display(out['2026_상대우선순위점수'].describe().round(3))

,기준점수,검증 선정 수,적중률(정밀도),2026 선정 수
0,0.3,439,0.499,453
1,0.4,271,0.601,280
2,0.5,166,0.759,191
3,0.6,110,0.845,126
4,0.7,61,0.934,90
5,0.8,26,1.000,43


2026 모델점수 분포:


,2026_상대우선순위점수
count,1548.000
mean,0.253
std,0.202
min,0.008
25%,0.107
50%,0.196
75%,0.335
max,0.953


In [ ]:
# 단일 선정규칙 일치 점검
assert out['우선순위'].eq(out['지원검토_2단계']).all()
assert out['우선순위'].eq('집중검토 후보').sum()==N_FINANCE
print('✅ 지도·CSV·분석표의 집중검토 후보 수:', N_FINANCE)

✅ 지도·CSV·분석표의 집중검토 후보 수: 155


## 발표·해석상의 주의사항

- **공식 집중검토 대상은 분류모델 상대우선순위 상위 10% 한 가지 규칙으로만 정합니다.**
- 상대우선순위점수는 보정된 발생확률이 아닙니다.
- 모델의 목표는 다음 해 G지수 기준 상권변화 위험이며, 실제 퇴출·임대료 상승·지원 자격이 아닙니다.
- 성능은 연도별 순위상관, 상위 10% 포착률, 적중 lift와 95% 신뢰구간으로 보고합니다.
- 회귀모델 유형은 공식 명단을 늘리지 않는 보조정보입니다.
- 자료미비 102개 상권은 저위험이 아니라 별도 확인 대상입니다.


## 6. 보조 회귀모델: 다음 해 G지수 수준과 상승폭

기존 이진 분류 외에 다음 해 G지수의 상대적 순위를 확인하기 위한 회귀모델을 사용합니다.

- **모델 선택:** 2024년 검증 결과만 사용
- **잠금 평가:** 선택된 모델을 바꾸지 않고 2025년 성능을 최종 확인
- **최종 예측:** 평가 종료 후 2022~2024년 전체로 재학습하여 2026년 산출
- 회귀 결과는 공식 후보를 추가하지 않고 분류모델 후보의 보조 검토에만 사용


In [ ]:
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import spearmanr

# G_다음은 기존 셀 11에서 상권 코드를 기준으로 t+1년에 연결한 값입니다.
assert {'상권_코드','연도','G지수','G_다음'}.issubset(idx.columns)
assert len(X)==len(idx) and X.index.equals(idx.index), '기존 특징행렬 X와 idx의 행이 다릅니다.'
assert not X.isna().any().any(), '회귀 입력에 결측값이 있습니다.'
assert idx.duplicated(['상권_코드','연도']).sum()==0

# 재현 가능한 고정 모델 (사후 튜닝 없음)
def reg_models():
    return {
        '전년_G유지': None,
        'Ridge': make_pipeline(StandardScaler(),Ridge(alpha=10.0)),
        'RandomForestRegressor': RandomForestRegressor(n_estimators=300,min_samples_leaf=8,
                                                       max_features=0.85,random_state=42,n_jobs=-1),
        'ExtraTreesRegressor': ExtraTreesRegressor(n_estimators=300,min_samples_leaf=8,
                                                   max_features=0.85,random_state=42,n_jobs=-1),
    }

def evaluate_continuous(y,hat):
    sp=spearmanr(y,hat).statistic if len(np.unique(hat))>1 else np.nan
    k=max(1,int(np.ceil(len(y)*0.10)))
    chosen=np.argsort(-hat,kind='stable')[:k]
    true_top=np.argsort(-y,kind='stable')[:k]
    return {'MAE':mean_absolute_error(y,hat),
            'RMSE':np.sqrt(mean_squared_error(y,hat)),
            'Spearman_rho':sp,
            '상위10%_실제G평균':np.mean(y[chosen]),
            '상위10%_실제상위10%_재현율':len(set(chosen)&set(true_top))/k}

valid_reg=idx['G_다음'].notna() & idx['연도'].between(2022,2024)
reg_rows=[]; prediction_rows=[]
for target in [2024,2025]:
    input_year=target-1
    tr=(idx['연도']<input_year) & valid_reg
    te=(idx['연도']==input_year) & valid_reg
    assert tr.sum()>100 and te.sum()>100, f'{target}년 학습·평가 표본 부족'
    ytr=idx.loc[tr,'G_다음'].to_numpy(dtype=float)
    yte=idx.loc[te,'G_다음'].to_numpy(dtype=float)
    for name,model in reg_models().items():
        if model is None:
            pred=idx.loc[te,'G지수'].to_numpy(dtype=float)
        else:
            model.fit(X.loc[tr],ytr)
            pred=model.predict(X.loc[te])
        met=evaluate_continuous(yte,pred)
        reg_rows.append({'예측대상연도':target,'모델':name,'훈련표본':tr.sum(),
                         '검증표본':te.sum(),**met})
        if name != '전년_G유지':
            part=idx.loc[te,['상권_코드','상권_코드_명','상권_구분_코드_명','연도','G지수']].copy()
            part['예측대상연도']=target
            part['모델']=name
            part['실제_다음해G']=yte
            part['예측_다음해G']=pred
            prediction_rows.append(part)

reg_eval=pd.DataFrame(reg_rows)
reg_oos=pd.concat(prediction_rows,ignore_index=True)
print('=== 다음 해 G지수 예측 검증 (미래 데이터 미사용) ===')
display(reg_eval.round(4))
reg_eval.to_csv(os.path.join(OUT_DIR,'2026_회귀모델_연도별검증.csv'),index=False,encoding='utf-8-sig')
reg_oos.to_csv(os.path.join(OUT_DIR,'2026_회귀모델_과거예측상세.csv'),index=False,encoding='utf-8-sig')


=== 다음 해 G지수 예측 검증 (미래 데이터 미사용) ===


,예측대상연도,모델,훈련표본,검증표본,MAE,RMSE,Spearman_rho,상위10%_실제G평균,상위10%_실제상위10%_재현율
0,2024,전년_G유지,1549,1546,0.3525,0.4739,0.4251,-0.9080,0.2968
1,2024,Ridge,1549,1546,0.2974,0.3818,0.5219,-0.7829,0.3548
2,2024,RandomForestRegressor,1549,1546,0.2948,0.3787,0.5365,-0.7594,0.3742
3,2024,ExtraTreesRegressor,1549,1546,0.2931,0.3765,0.5469,-0.7414,0.3806
4,2025,전년_G유지,3095,1547,0.3455,0.4718,0.4569,-0.8531,0.3484
5,2025,Ridge,3095,1547,0.2875,0.3772,0.5277,-0.7203,0.4323
6,2025,RandomForestRegressor,3095,1547,0.2816,0.3665,0.5473,-0.6637,0.4645
7,2025,ExtraTreesRegressor,3095,1547,0.2786,0.3644,0.5569,-0.6568,0.4710


### 회귀모델 선택과 잠금 평가

2024년 검증 MAE만으로 모델을 선택하고, 선택 결과를 변경하지 않은 상태에서 2025년을 잠금 평가합니다. 단순 `전년 G 유지`보다 2024년에 낫지 않으면 기준모델을 사용합니다. 2025년 결과는 모델 재선택에 사용하지 않습니다.


In [ ]:
agg=reg_eval.groupby('모델').agg(
    평균_MAE=('MAE','mean'),평균_RMSE=('RMSE','mean'),
    평균_순위상관=('Spearman_rho','mean'),
    평균_상위10재현율=('상위10%_실제상위10%_재현율','mean')
).sort_values('평균_MAE')
display(agg.round(4))

# 2024년만 모델 선택에 사용하고 2025년은 잠금 평가로 남긴다.
selection_2024=reg_eval.loc[reg_eval['예측대상연도'].eq(2024)].set_index('모델').sort_values('MAE')
candidate_name=selection_2024.index[0]
if selection_2024.loc[candidate_name,'MAE'] < selection_2024.loc['전년_G유지','MAE']:
    chosen_name=candidate_name
else:
    chosen_name='전년_G유지'
locked_test=reg_eval.loc[(reg_eval['예측대상연도'].eq(2025)) & reg_eval['모델'].eq(chosen_name)].iloc[0]
chosen_mae=float(locked_test['MAE'])
print(f'2024년 선택 모델: {chosen_name}')
print(f'2025년 잠금 평가: MAE={locked_test["MAE"]:.4f}, '
      f'Spearman={locked_test["Spearman_rho"]:.4f}, '
      f'상위10% 포착률={locked_test["상위10%_실제상위10%_재현율"]:.4f}')

# 평가 종료 후 2022~2024년 전체 자료로 2026년 예측
model=reg_models()[chosen_name]
train_all=(idx['연도'].between(2022,2024)) & valid_reg
mask25=idx['연도'].eq(2025)
assert int(mask25.sum())==len(out),'2025년 예측 대상 개수와 기존 out 개수 불일치'
if model is None:
    predicted=idx.loc[mask25,'G지수'].to_numpy(dtype=float)
else:
    model.fit(X.loc[train_all],idx.loc[train_all,'G_다음'].to_numpy(dtype=float))
    predicted=model.predict(X.loc[mask25])

reg2026=idx.loc[mask25,['상권_코드','G지수']].copy()
reg2026['2026_예측G']=predicted
reg2026=reg2026[['상권_코드','2026_예측G']]
out=out.drop(columns=['2026_예측G','2026_예측G변화','2026_회귀보조유형'],errors='ignore').merge(
    reg2026,on='상권_코드',how='left',validate='one_to_one')
assert out['2026_예측G'].notna().all()
out['2026_예측G변화']=out['2026_예측G']-out['G지수']
out['회귀선정모델']=chosen_name
out['회귀잠금평가_MAE']=float(locked_test['MAE'])
out['회귀잠금평가_순위상관']=float(locked_test['Spearman_rho'])

# 회귀 순위는 공식 후보를 추가하지 않는 보조정보다.
n_focus=max(1,int(np.ceil(len(out)*.10)))
high_g_ids=set(out.sort_values(['2026_예측G','상권_코드'],ascending=[False,True]).head(n_focus)['상권_코드'])
out['2026_예측G_상위10']=out['상권_코드'].isin(high_g_ids)
rising_pool=out['G지수']<0
rising_candidates=out.loc[rising_pool].sort_values(['2026_예측G변화','상권_코드'],ascending=[False,True])
n_rise=max(1,int(np.ceil(len(rising_candidates)*.10)))
rise_ids=set(rising_candidates.head(n_rise)['상권_코드'])
out['2026_상승상위10']=out['상권_코드'].isin(rise_ids) & (out['2026_예측G변화']>0)
out['2026_회귀보조유형']=np.select([
    out['2026_예측G_상위10'] & (out['G지수']>=0),
    out['2026_예측G_상위10'],
    out['2026_상승상위10']
],[
    '현재G≥0·예측G순위 상위',
    '예측G순위 상위',
    '예측상승폭 상위'
],default='일반')

# 공식 여부는 분류모델의 단일 규칙을 그대로 사용한다.
out['2026_공식집중검토']=out['지원검토_2단계'].eq('집중검토 후보')
assert int(out['2026_공식집중검토'].sum())==N_FINANCE
print('\n=== 회귀 보조유형 (공식 후보 추가 없음) ===')
display(out['2026_회귀보조유형'].value_counts().rename_axis('보조유형').reset_index(name='개수'))

cols=['서울상권_우선순위','지원검토_2단계','2026_회귀보조유형','자치구_코드_명','상권_코드_명',
      '2026_상대우선순위점수','G지수','2026_예측G','2026_예측G변화','점포','폐업']
display(out.loc[out['2026_공식집중검토']].sort_values('서울상권_우선순위')[cols].head(25).round(3))

handoff=out.sort_values(['2026_공식집중검토','서울상권_우선순위'],ascending=[False,True])
handoff.to_csv(os.path.join(OUT_DIR,'2026_전체상권_통합우선순위.csv'),index=False,encoding='utf-8-sig')
handoff.loc[handoff['2026_공식집중검토']].to_csv(
    os.path.join(OUT_DIR,'2026_공식집중검토_통합지표.csv'),index=False,encoding='utf-8-sig')
out.to_csv(os.path.join(OUT_DIR,'2026_회귀보조지표_전체상권.csv'),index=False,encoding='utf-8-sig')
agg.to_csv(os.path.join(OUT_DIR,'2026_회귀모델_비교.csv'),encoding='utf-8-sig')


,평균_MAE,평균_RMSE,평균_순위상관,평균_상위10재현율
모델,,,,
ExtraTreesRegressor,0.2858,0.3705,0.5519,0.4258
RandomForestRegressor,0.2882,0.3726,0.5419,0.4194
Ridge,0.2924,0.3795,0.5248,0.3935
전년_G유지,0.3490,0.4729,0.4410,0.3226


2024년 선택 모델: ExtraTreesRegressor
2025년 잠금 평가: MAE=0.2786, Spearman=0.5569, 상위10% 포착률=0.4710

=== 회귀 보조유형 (공식 후보 추가 없음) ===


,보조유형,개수
0,일반,1243
1,예측상승폭 상위,150
2,예측G순위 상위,146
3,현재G≥0·예측G순위 상위,9


,서울상권_우선순위,지원검토_2단계,2026_회귀보조유형,자치구_코드_명,상권_코드_명,2026_상대우선순위점수,G지수,2026_예측G,2026_예측G변화,점포,폐업
0,1,집중검토 후보,예측G순위 상위,용산구,숙대입구,0.953,-0.373,-0.516,-0.143,536.50,67
1,2,집중검토 후보,예측G순위 상위,마포구,DMC(디지털미디어시티),0.951,-0.441,-0.579,-0.138,874.25,126
2,3,집중검토 후보,예측G순위 상위,강동구,강일동주민센터,0.942,-0.723,-0.466,0.257,209.00,31
3,4,집중검토 후보,예측G순위 상위,강서구,양천향교역 7번,0.932,-0.114,-0.340,-0.226,388.50,70
4,5,집중검토 후보,예측G순위 상위,동작구,중앙대학교,0.931,-0.282,-0.561,-0.279,213.75,25
5,6,집중검토 후보,예측G순위 상위,동대문구,경희대학교(경희대),0.922,-0.441,-0.579,-0.138,411.25,62
6,7,집중검토 후보,현재G≥0·예측G순위 상위,노원구,서울과학기술대,0.921,0.306,-0.306,-0.612,91.00,21
7,8,집중검토 후보,예측G순위 상위,금천구,독산역 1번 출입구,0.917,-0.282,-0.294,-0.012,69.00,8
8,9,집중검토 후보,예측G순위 상위,강동구,고덕 골목형상점가,0.915,-0.575,-0.608,-0.033,183.00,28
9,10,집중검토 후보,예측G순위 상위,노원구,경춘선숲길 우측,0.910,-0.484,-0.594,-0.110,351.75,44


### 회귀 위험순위의 과거 검증

선택된 회귀모델이 다음 해 실제 G지수 상위 10%를 얼마나 포착하는지 평가합니다. `상승폭 상위`는 보조분석이며 공식 후보 선정에는 사용하지 않습니다.


In [ ]:
# 선택한 모델의 과거 홀드아웃 예측만 사용 (회귀 성능과 별도)
if chosen_name=='전년_G유지':
    selected_oos=[]
    for year in [2024,2025]:
        t=idx[(idx['연도']==year-1)&valid_reg].copy()
        t['예측대상연도']=year
        t['예측_다음해G']=t['G지수']
        t['실제_다음해G']=t['G_다음']
        selected_oos.append(t)
    se=pd.concat(selected_oos,ignore_index=True)
else:
    se=reg_oos.loc[reg_oos['모델']==chosen_name].copy()
verify=[]
for year,part in se.groupby('예측대상연도'):
    p=part.copy()
    k=max(1,int(np.ceil(len(p)*.10)))
    rank_pred=p.nlargest(k,'예측_다음해G')['상권_코드']
    rank_true=p.nlargest(k,'실제_다음해G')['상권_코드']
    all_hit=len(set(rank_pred)&set(rank_true))/k
    candidate=p.loc[p['G지수']<0].copy()
    kr=max(1,int(np.ceil(len(candidate)*.10)))
    candidate['예측상승']=candidate['예측_다음해G']-candidate['G지수']
    candidate['실제상승']=candidate['실제_다음해G']-candidate['G지수']
    rising_pred=candidate.nlargest(kr,'예측상승')['상권_코드']
    rising_true=candidate.nlargest(kr,'실제상승')['상권_코드']
    rise_hit=len(set(rising_pred)&set(rising_true))/kr
    verify.append({'검증연도':year,'회귀선정모델':chosen_name,'검증상권수':len(p),
                   '실제G상위10_포착률':all_hit,'현재G0미만_상권수':len(candidate),
                   '예측상승폭상위10_포착률':rise_hit,
                   '무작위상위10_기대재현율_참고':0.1})
verify=pd.DataFrame(verify)
print('=== 회귀 보조순위의 실제 G지수 기준 과거 검증 ===')
display(verify.round(4))
verify.to_csv(os.path.join(OUT_DIR,'검증_회귀보조순위_과거성능.csv'),index=False,encoding='utf-8-sig')


=== 회귀 보조순위의 실제 G지수 기준 과거 검증 ===


,검증연도,회귀선정모델,검증상권수,실제G상위10_포착률,현재G0미만_상권수,예측상승폭상위10_포착률,무작위상위10_기대재현율_참고
0,2024,ExtraTreesRegressor,1546,0.3806,1531,0.2922,0.1
1,2025,ExtraTreesRegressor,1547,0.4710,1523,0.2288,0.1


### 순위상관 95% 신뢰구간과 운영 적합성 기준

상관계수의 점추정치만 보고 판단하지 않도록 상권 단위 부트스트랩 95% 신뢰구간을 계산합니다.

이 프로젝트의 **사전 운영기준**은 다음과 같습니다. 이는 보편적인 학술 기준이 아니라 현장검토 우선순위 도구로 쓰기 위한 내부 기준입니다.

- 두 검증연도 모두 Spearman 순위상관 95% 신뢰구간 하한이 0.40 이상
- 실제 G 상위 10% 포착 lift가 무작위 대비 2배 이상

두 조건을 모두 충족할 때만 `우선순위 선별용으로 사용 가능`으로 표시합니다.


In [ ]:
# 선택 모델의 연도별 순위상관 신뢰구간과 상위 10% 포착 lift
rng_corr=np.random.default_rng(20261027)
corr_rows=[]
for year,part in se.groupby('예측대상연도'):
    p=part.dropna(subset=['실제_다음해G','예측_다음해G']).reset_index(drop=True)
    actual=p['실제_다음해G'].to_numpy(float)
    pred=p['예측_다음해G'].to_numpy(float)
    rho=float(spearmanr(actual,pred).statistic)
    boot_rho=[]
    for _ in range(1000):
        take=rng_corr.integers(0,len(p),len(p))
        if np.unique(actual[take]).size>1 and np.unique(pred[take]).size>1:
            boot_rho.append(float(spearmanr(actual[take],pred[take]).statistic))
    lo,hi=np.quantile(boot_rho,[.025,.975])
    k=max(1,int(np.ceil(len(p)*.10)))
    pred_top=set(np.argsort(-pred,kind='stable')[:k])
    true_top=set(np.argsort(-actual,kind='stable')[:k])
    capture=len(pred_top & true_top)/k
    lift=capture/0.10
    corr_rows.append({'검증연도':year,'선정모델':chosen_name,'표본수':len(p),
                      'Spearman_rho':rho,'rho_CI_2.5%':lo,'rho_CI_97.5%':hi,
                      '실제G상위10_포착률':capture,'무작위대비_포착lift':lift,
                      '연도별_운영기준통과':(lo>=0.40) and (lift>=2.0)})
corr_validation=pd.DataFrame(corr_rows)
overall_pass=bool(corr_validation['연도별_운영기준통과'].all())
print('=== 상권변화 위험 우선순위 운영 적합성 ===')
display(corr_validation.round(4))
print('최종 판정:', '우선순위 선별용으로 사용 가능' if overall_pass else '운영기준 미충족 — 탐색용으로만 사용')
corr_validation.to_csv(os.path.join(OUT_DIR,'검증_순위상관_95CI_운영판정.csv'),index=False,encoding='utf-8-sig')


=== 상권변화 위험 우선순위 운영 적합성 ===


,검증연도,선정모델,표본수,Spearman_rho,rho_CI_2.5%,rho_CI_97.5%,실제G상위10_포착률,무작위대비_포착lift,연도별_운영기준통과
0,2024,ExtraTreesRegressor,1546,0.5469,0.5116,0.5824,0.3806,3.8065,True
1,2025,ExtraTreesRegressor,1547,0.5569,0.5194,0.5948,0.4710,4.7097,True


최종 판정: 우선순위 선별용으로 사용 가능


## 7. 회귀 보조지표 지도

이 지도는 공식 후보를 추가하는 지도가 아닙니다. 공식 후보 지도는 앞 절의 분류모델 상위 10% 지도입니다.

- **진홍색:** 현재 G≥0이면서 예측 G순위 상위
- **주황색:** 예측 G순위 상위
- **보라색:** 예측 상승폭 상위
- **회색:** 일반

중심점 표시이며 실제 상권 경계가 아닙니다.


In [ ]:
import folium
from pyproj import Transformer
from folium.plugins import Fullscreen
from html import escape

geo_tf=Transformer.from_crs('EPSG:5181','EPSG:4326',always_xy=True)
out['경도'],out['위도']=geo_tf.transform(pd.to_numeric(out['엑스좌표_값'],errors='coerce').to_numpy(),
                                       pd.to_numeric(out['와이좌표_값'],errors='coerce').to_numpy())
valid_geo=out['위도'].between(37.4,37.72)&out['경도'].between(126.7,127.25)
print('서울 범위 좌표:',int(valid_geo.sum()),'/',len(out))
if int(valid_geo.sum())==0: raise ValueError('지도 좌표가 유효하지 않습니다. 좌표계 확인 필요')

palette={
    '현재G≥0·예측G순위 상위':('#b91c1c',7,True),
    '예측G순위 상위':('#ea580c',7,True),
    '예측상승폭 상위':('#7c3aed',6,True),
    '일반':('#9ca3af',2,False),
}
m2=folium.Map(location=[37.5665,126.978],zoom_start=11,tiles='CartoDB positron',control_scale=True)
layers={k:folium.FeatureGroup(name=f'{k} ({int((out["2026_회귀보조유형"]==k).sum())}개)',show=v[2]) for k,v in palette.items()}
for _,r in out.loc[valid_geo].iterrows():
    kind=r['2026_회귀보조유형'];col,rad,_=palette[kind]
    info=(f'<b>{escape(str(r["상권_코드_명"]))}</b><br>{escape(str(r["자치구_코드_명"]))}<br>'
          f'유형: {escape(kind)}<br>2025 G: {r["G지수"]:.3f}<br>'
          f'2026 예측 G: {r["2026_예측G"]:.3f}<br>예측 변화: {r["2026_예측G변화"]:+.3f}<br>'
          f'검증 MAE 평균: {chosen_mae:.3f}<br>'
          '<small>공식 선정이 아닌 회귀 보조정보</small>')
    folium.CircleMarker([r['위도'],r['경도']],radius=rad,color=col,fill=True,fill_color=col,
                        fill_opacity=0.85 if kind!='일반' else 0.25,weight=1,
                        tooltip=f'{r["상권_코드_명"]}: {kind}',
                        popup=folium.Popup(info,max_width=360)).add_to(layers[kind])
for k in ['일반','예측상승폭 상위','예측G순위 상위','현재G≥0·예측G순위 상위']:
    layers[k].add_to(m2)
folium.LayerControl(collapsed=False).add_to(m2)
Fullscreen().add_to(m2)
legend="""<div style="position:fixed;bottom:20px;left:20px;z-index:9999;background:white;
border:1px solid #aaa;border-radius:8px;padding:12px;font:12px/1.7 sans-serif;max-width:310px">
<b>2026 회귀 보조지표</b><br>
<span style="color:#b91c1c">●</span> 현재G≥0·예측G순위 상위<br>
<span style="color:#ea580c">●</span> 예측G순위 상위<br>
<span style="color:#7c3aed">●</span> 예측상승폭 상위<br>
<small>공식 후보 추가 선정에 사용하지 않음. 중심점 기준 지도.</small>
</div>"""
m2.get_root().html.add_child(folium.Element(legend))
new_map=os.path.join(OUT_DIR,'2026_회귀보조지표_지도.html')
m2.save(new_map)
print('지도:',new_map)
print('팀 전달 CSV:',os.path.join(OUT_DIR,'2026_공식집중검토_통합지표.csv'))
display(m2)


서울 범위 좌표: 1548 / 1548


/usr/local/lib/python3.13/dist-packages/folium/raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore


지도: /content/drive/MyDrive/젠트리피케이션_2026_결과/2026_회귀보조지표_지도.html
팀 전달 CSV: /content/drive/MyDrive/젠트리피케이션_2026_결과/2026_공식집중검토_통합지표.csv


In [ ]:
# ── 10. 선정 비율별 적중률 (상위 3% / 5% / 10%) — 단독 실행용 ─────────────────
# 필요 변수: a, idx, W, V6, OUT_DIR (앞의 1~7번 셀을 실행하면 생깁니다. 9번 셀은 필요 없음)
# 탐색적 분석: 비율을 고르는 기준은 '적중률이 가장 높은 것'이 아니라 현장 조사 인력 규모입니다.
import statsmodels.formula.api as smf

MIN_STORE, HIGHQ = 10, 0.75

# 0) 상권×연도 기본표 (입력)
o = a[['상권_코드', '연도', '점포', '폐업']].copy()
o['폐업률'] = np.where(o['점포'] > 0, o['폐업'] / o['점포'] * 100, np.nan)
code_cols = [v + '_코드' for v in V6]
base = idx[['상권_코드', '연도', '상권_코드_명', '상권_구분_코드_명', '자치구_코드_명'] + code_cols].merge(
    o[['상권_코드', '연도', '점포', '폐업률']], on=['상권_코드', '연도'], how='left', validate='one_to_one')
keep = [v for v in V6 if v != '창폐업']
w = np.array([W[v] for v in keep], dtype=float); w = w / w.sum()
base['G5'] = sum(wi * base[v + '_코드'] for wi, v in zip(w, keep))
base['G5_z'] = base.groupby('연도')['G5'].transform(lambda s: (s - s.mean()) / s.std())
base['close_now'] = base['폐업률']
base['logstore'] = np.log1p(base['점포'])
base['typ'] = base['상권_구분_코드_명'].astype(str)

FORM = {'단순기준': 'y ~ close_now + logstore + C(typ)',
        '지수포함': 'y ~ close_now + logstore + C(typ) + G5_z'}

def make(h):
    """입력연도 t → t+h년 폐업률을 붙인 학습/검증용 표"""
    t = o[['상권_코드', '연도', '폐업률']].rename(columns={'폐업률': 'y'})
    t['연도'] = t['연도'] - h
    d = base.merge(t, on=['상권_코드', '연도'], how='inner')
    d = d[d['점포'] >= MIN_STORE].dropna(subset=['y', 'close_now', 'G5_z', 'logstore']).copy()
    d['high'] = (d['y'] >= d.groupby('연도')['y'].transform(lambda s: s.quantile(HIGHQ))).astype(int)
    return d


CUTS = [0.03, 0.05, 0.10]
rng10 = np.random.default_rng(10)
rows10 = []
for h in [1, 2]:
    d = make(h)
    tys = sorted(y for y in d['연도'].unique() if (d['연도'] < y).any())
    parts = []
    for ty in tys:
        tr, te = d[d['연도'] < ty], d[d['연도'] == ty].copy()
        te['pred'] = smf.ols(FORM['단순기준'], data=tr).fit().predict(te)
        te['rk'] = te['pred'].rank(ascending=False, method='first') / len(te)   # 상위 몇 %인지
        parts.append(te)
    det = pd.concat(parts, ignore_index=True)
    groups = {c: g.index.to_numpy() for c, g in det.groupby('상권_코드')}
    codes = np.array(list(groups))
    for cut in CUTS:
        sel = det['rk'] <= cut
        n, hit = int(sel.sum()), int(det.loc[sel, 'high'].sum())
        prec, base_rate = hit / n, det['high'].mean()
        recall = hit / det['high'].sum()
        bs = []
        for _ in range(500):
            s = det.loc[np.concatenate([groups[c] for c in rng10.choice(codes, len(codes))])]
            m = s['rk'] <= cut
            if m.any(): bs.append(s.loc[m, 'high'].mean())
        lo, hi = np.percentile(bs, [2.5, 97.5])
        rows10.append({'예측시점': f'{h}년 뒤', '선정비율': f'상위 {cut:.0%}', '선정수(연평균)': round(n / len(tys)),
                       '적중률': prec, '95%CI': f'{lo:.2f}~{hi:.2f}', '무작위기대': base_rate,
                       '배율': prec / base_rate, '포착률': recall, '헛지원비율': 1 - prec})
cut_tab = pd.DataFrame(rows10)
print('=== 선정 비율별 적중률 (단순 폐업률 기준, 시간 순서 검증) ===')
display(cut_tab.round(3))
cut_tab.to_csv(os.path.join(OUT_DIR, '지원선정_비율별_적중률.csv'), index=False, encoding='utf-8-sig')
print('· 적중률: 선정한 상권 중 실제 폐업 고위험(그 해 상위 25%) 비율 / 포착률: 전체 고위험 중 찾아낸 비율')
print('· 배율: 무작위로 고를 때 대비 몇 배 / 헛지원비율 = 1 − 적중률')

=== 선정 비율별 적중률 (단순 폐업률 기준, 시간 순서 검증) ===


,예측시점,선정비율,선정수(연평균),적중률,95%CI,무작위기대,배율,포착률,헛지원비율
0,1년 뒤,상위 3%,46,0.533,0.43~0.64,0.25,2.129,0.063,0.467
1,1년 뒤,상위 5%,77,0.545,0.46~0.63,0.25,2.180,0.108,0.455
2,1년 뒤,상위 10%,155,0.513,0.46~0.57,0.25,2.050,0.205,0.487
3,2년 뒤,상위 3%,46,0.500,0.36~0.62,0.25,2.000,0.059,0.500
4,2년 뒤,상위 5%,77,0.519,0.41~0.63,0.25,2.078,0.103,0.481
5,2년 뒤,상위 10%,155,0.490,0.41~0.57,0.25,1.961,0.196,0.510


· 적중률: 선정한 상권 중 실제 폐업 고위험(그 해 상위 25%) 비율 / 포착률: 전체 고위험 중 찾아낸 비율
· 배율: 무작위로 고를 때 대비 몇 배 / 헛지원비율 = 1 − 적중률


## 분석 결과 해석과 전달 기준

1. **공식 선정:** `2026_상대우선순위점수` 상위 10%만 집중검토 후보입니다.
2. **핵심 성능:** 다음 해 G지수에 대한 연도별 순위상관, 95% 신뢰구간, 상위 10% 포착 lift로 판단합니다.
3. **회귀 보조정보:** `2026_예측G`, `2026_예측G변화`, `2026_회귀보조유형`은 공식 후보를 늘리지 않습니다.
4. **운영 판정:** 두 검증연도 모두 사전 운영기준을 통과할 때만 우선순위 선별용으로 사용합니다.
5. **자료미비:** `2026_자료미비_별도검토상권.csv`의 상권은 안전하다는 뜻이 아닙니다.
6. **범위:** 본 결과는 G지수로 정의한 상권변화 위험의 상대적 우선순위이며 실제 피해·임대료 상승·금융지원 자격을 의미하지 않습니다.

### 최종 전달 파일

- `2026_공식집중검토_통합지표.csv`: 공식 상위 10% 후보와 회귀 보조지표
- `2026_전체상권_통합우선순위.csv`: 평가 가능한 전체 상권
- `2026_자료미비_별도검토상권.csv`: 자료미비 별도 확인 대상
- `검증_순위상관_95CI_운영판정.csv`: 연도별 상관관계와 운영기준 판정
- `2026_집중검토상권_지도.html`: 공식 후보 지도
- `2026_회귀보조지표_지도.html`: 보조 회귀 지도
